In [9]:
import sys
sys.path.append(r"D:\CST2020\AMD64\python_cst_libraries")
import cst.interface
import cst.results
import numpy as np
import pandas as pd
import os

In [ ]:
path =r"F:\metasurface_zwj\第三章\3.1超表面数据集\wanzhengpara.csv"
arr = pd.read_csv(path, header=None)
arr = arr.values.tolist()


# 建立仿真环境与模型
cst = cst.interface.DesignEnvironment()
mws = cst.new_mws()
modeler = mws.modeler

p = 15.4                   # 单元周期    mm
h1 = 0.01                  # 底板金属厚度 mm， 论文中是0.018mm, 怀疑论文中复制错误
h2 = 1.2                   # 中间介质层厚度
h3 = 0.035                 # 顶层金属厚度 mm, 论文中是0.018mm
modeler.add_to_history('StoreParameter', 'MakeSureParameterExists("theta","0")')
modeler.add_to_history('StoreParameter', 'MakeSureParameterExists("phi","0")')
modeler.add_to_history('StoreParameter', f'MakeSureParameterExists("p","{p}")')
modeler.add_to_history('StoreParameter', f'MakeSureParameterExists("h1","{h1}")')
modeler.add_to_history('StoreParameter', f'MakeSureParameterExists("h2","{h2}")')
modeler.add_to_history('StoreParameter', f'MakeSureParameterExists("h3","{h3}")')


# 全局初始化
line_break = '\n'       # 用来换行
# set the units 设置单位
sCommand = ['With Units',
            '.Geometry "mm"',
            '.Frequency "GHz"',
            '.Voltage "V"',
            '.Resistance "Ohm"',
            '.Inductance "H"',
            '.TemperatureUnit  "Kelvin"',
            '.Time "s"',
            '.Current "A"',
            '.Conductance "Siemens"',
            '.Capacitance "F"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history('set units', sCommand)

# 设置频率范围
sCommand = 'Solver.FrequencyRange "5", "12"'
modeler.add_to_history('set the frequency range', sCommand)

sCommand = 'Plot.DrawBox True'
modeler.add_to_history('set DrawBox', sCommand)

# set the Background
sCommand = ['With Background',
            '.Type "Normal"',
            '.Epsilon "1.0"',
            '.Mu "1.0"',
            '.Rho "1.204"',
            '.ThermalType "Normal"',
            '.ThermalConductivity "0.026"',
            '.HeatCapacity "1.005"',
            '.XminSpace "0.0"',
            '.XmaxSpace "0.0"',
            '.YminSpace "0.0"',
            '.YmaxSpace "0.0"',
            '.ZminSpace "0.0"',
            '.ZmaxSpace "0.0"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history("set background", sCommand)

# define Floquet port boundaries
sCommand = ['With FloquetPort',
            '.Reset',
            '.SetDialogTheta "0"',
            '.SetDialogPhi "0"',
            '.SetSortCode "+beta/pw"',
            '.SetCustomizedListFlag "False"',
            '.Port "Zmin"',
            '.SetNumberOfModesConsidered "2"',
            '.Port "Zmax"',
            '.SetNumberOfModesConsidered "2"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history("set Floquet", sCommand)

# define boundaries
sCommand = ['With Boundary',
            '.Xmin "unit cell"',
            '.Xmax "unit cell"',
            '.Ymin "unit cell"',
            '.Ymax "unit cell"',
            '.Zmin "expanded open"',
            '.Zmax "expanded open"',
            '.Xsymmetry "none"',
            '.Ysymmetry "none"',
            '.Zsymmetry "none"',
            '.XPeriodicShift "0.0"',
            '.YPeriodicShift "0.0"',
            '.ZPeriodicShift "0.0"',
            '.PeriodicUseConstantAngles "False"',
            '.SetPeriodicBoundaryAngles "theta", "phi"',
            '.SetPeriodicBoundaryAnglesDirection "inward"',
            '.UnitCellFitToBoundingBox "True"',
            '.UnitCellDs1 "0.0"',
            '.UnitCellDs2 "0.0"',
            '.UnitCellAngle "90.0"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history("set boundaries", sCommand)

# set tet mesh as default
sCommand = ['With Mesh',
            '.MeshType "Tetrahedral"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history("set tet mesh as default", sCommand)

# FD solver excitation with incoming plane wave at Zmax
sCommand = ['With FDSolver',
            '.Reset',
            '.Stimulation "List", "List"',
            '.ResetExcitationList',
            '.AddToExcitationList "Zmax", "TE(0,0);TM(0,0)"',
            '.LowFrequencyStabilization "False"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history("set solver", sCommand)

sCommand = ['With MeshSettings',
            '.SetMeshType "Tet"',
            '.Set "Version", 1%',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history("MeshSettings", sCommand)

sCommand = ['With Mesh',
            '.MeshType "Tetrahedral"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history("Mesh", sCommand)

# set the solver type
sCommand = 'ChangeSolverType("HF Frequency Domain")'
modeler.add_to_history('set solver type', sCommand)
# 全局数据初始化结束

# 定义材料
# 退火铜
sCommand = ['With Material',
            '.Reset',
            '.Name "Copper (annealed)"',
            '.Folder ""',
            '.FrqType "static"',
            '.Type "Normal"',
            '.SetMaterialUnit "Hz", "mm"',
            '.Epsilon "1"',
            '.Mu "1.0"',
            '.Kappa "5.8e+007"',
            '.TanD "0.0"',
            '.TanDFreq "0.0"',
            '.TanDGiven "False"',
            '.TanDModel "ConstTanD"',
            '.KappaM "0"',
            '.TanDM "0.0"',
            '.TanDMFreq "0.0"',
            '.TanDMGiven "False"',
            '.TanDMModel "ConstTanD"',
            '.DispModelEps "None"',
            '.DispModelMu "None"',
            '.DispersiveFittingSchemeEps "Nth Order"',
            '.DispersiveFittingSchemeMu "Nth Order"',
            '.UseGeneralDispersionEps "False"',
            '.UseGeneralDispersionMu "False"',
            '.FrqType "all"',
            '.Type "Lossy metal"',
            '.SetMaterialUnit "GHz", "mm"',
            '.Mu "1.0"',
            '.Kappa "5.8e+007"',
            '.Rho "8930.0"',
            '.ThermalType "Normal"',
            '.ThermalConductivity "401.0"',
            '.SpecificHeat "390", "J/K/kg"',
            '.MetabolicRate "0"',
            '.BloodFlow "0"',
            '.VoxelConvection "0"',
            '.MechanicsType "Isotropic"',
            '.YoungsModulus "120"',
            '.PoissonsRatio "0.33"',
            '.ThermalExpansionRate "17"',
            '.Colour "1", "1", "0"',
            '.Wireframe "False"',
            '.Reflection "False"',
            '.Allowoutline "True"',
            '.Transparentoutline "False"',
            '.Transparency "0"',
            '.Create',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history('define Copper(annealed)', sCommand)

# FR-4材料(有损耗)
sCommand = ['With Material',
            '.Reset',
            '.Name "FR-4 (lossy)"',
            '.Folder ""',
            '.FrqType "all"',
            '.Type "Normal"',
            '.SetMaterialUnit "GHz", "mm"',
            '.Epsilon "4.3"',
            '.Mu "1.0"',
            '.Kappa "0.0"',
            '.TanD "0.025"',
            '.TanDFreq "10.0"',
            '.TanDGiven "True"',
            '.TanDModel "ConstTanD"',
            '.KappaM "0.0"',
            '.TanDM "0.0"',
            '.TanDMFreq "0.0"',
            '.TanDMGiven "False"',
            '.TanDMModel "ConstKappa"',
            '.DispModelEps "None"',
            '.DispModelMu "None"',
            '.DispersiveFittingSchemeEps "General 1st"',
            '.DispersiveFittingSchemeMu "General 1st"',
            '.UseGeneralDispersionEps "False"',
            '.UseGeneralDispersionMu "False"',
            '.Rho "0.0"',
            '.ThermalType "Normal"',
            '.ThermalConductivity "0.3"',
            '.SetActiveMaterial "all"',
            '.Colour "0.94", "0.82", "0.76"',
            '.Wireframe "False"',
            '.Transparency "0"',
            '.Create',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history('define FR-4(lossy)', sCommand)

sCommand = ['With Mesh',
            '.MeshType "Tetrahedral"',
            '.SetCreator "High Frequency"',
            'End With',
            'With MeshSettings',
            '.SetMeshType "Tet"',
            ' .Set "Version", 1%',
            #'MAX CELL - WAVELENGTH REFINEMENT
            '.Set "StepsPerWaveNear", "4"',
            '.Set "StepsPerWaveFar", "4"',
            '.Set "PhaseErrorNear", "0.02"',
            '.Set "PhaseErrorFar", "0.02"',
            '.Set "CellsPerWavelengthPolicy", "automatic" ',
            #'MAX CELL - GEOMETRY REFINEMENT
            '.Set "StepsPerBoxNear", "10"',
            '.Set "StepsPerBoxFar", "1"',
            '.Set "ModelBoxDescrNear", "maxedge"',
            '.Set "ModelBoxDescrFar", "maxedge" ',
            #'MIN CELL
            '.Set "UseRatioLimit", "0"',
            '.Set "RatioLimit", "100"',
            '.Set "MinStep", "0" ',
            #'MESHING METHOD
            '.SetMeshType "Unstr"',
            '.Set "Method", "0"',
            'End With',
            'With MeshSettings',
            '.SetMeshType "Tet"',
            '.Set "CurvatureOrder", "1"',
            '.Set "CurvatureOrderPolicy", "automatic"',
            '.Set "CurvRefinementControl", "NormalTolerance"',
            '.Set "NormalTolerance", "22.5"',
            '.Set "SrfMeshGradation", "1.5"',
            '.Set "SrfMeshOptimization", "1"',
            'End With',
            'With MeshSettings',
            '.SetMeshType "Unstr"',
            '.Set "UseMaterials",  "1"',
            '.Set "MoveMesh", "0"',
            'End With',
            'With MeshSettings',
            '.SetMeshType "All"',
            '.Set "AutomaticEdgeRefinement",  "0"',
            'End With',
            'With MeshSettings',
            '.SetMeshType "Tet"',
            '.Set "UseAnisoCurveRefinement", "1"',
            '.Set "UseSameSrfAndVolMeshGradation", "1"',
            '.Set "VolMeshGradation", "1.5"',
            '.Set "VolMeshOptimization", "1"',
            'End With',
            'With MeshSettings',
            '.SetMeshType "Unstr"',
            '.Set "SmallFeatureSize", "0"',
            '.Set "CoincidenceTolerance", "1e-06"',
            '.Set "SelfIntersectionCheck", "1"',
            '.Set "OptimizeForPlanarStructures", "0"',
            'End With',
            'With Mesh',
            '.SetParallelMesherMode "Tet", "maximum"',
            '.SetMaxParallelMesherThreads "Tet", "1"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history('set mesh properties(Tetrahedral)', sCommand)

sCommand = ['With MeshAdaption3D',
            '.SetType "HighFrequencyTet"',
            '.SetAdaptionStrategy "ExpertSystem"',
            '.MinPasses "3"',
            '.MaxPasses "8"',
            '.ClearStopCriteria',
            '.MaxDeltaS "0.02"',
            '.NumberOfDeltaSChecks "1"',
            '.EnableInnerSParameterAdaptation "True"',
            '.PropagationConstantAccuracy "0.005"',
            '.NumberOfPropConstChecks "2"',
            '.EnablePortPropagationConstantAdaptation "True"',
            '.RemoveAllUserDefinedStopCriteria',
            '.AddStopCriterion "All S-Parameters", "0.02", "1", "True"',
            '.AddStopCriterion "Reflection S-Parameters", "0.02", "1", "False"',
            '.AddStopCriterion "Transmission S-Parameters", "0.02", "1", "False"',
            '.AddStopCriterion "Portmode kz/k0", "0.005", "2", "True"',
            '.AddStopCriterion "All Probes", "0.05", "2", "False"',
            '.AddSParameterStopCriterion "True", "", "", "0.02", "1", "False"',
            '.MinimumAcceptedCellGrowth "0.5"',
            '.RefThetaFactor ""',
            '.SetMinimumMeshCellGrowth "5"',
            '.ErrorEstimatorType "Automatic"',
            '.RefinementType "Automatic"',
            '.SnapToGeometry "True"',
            '.SubsequentChecksOnlyOnce "False"',
            '.WavelengthBasedRefinement "True"',
            '.EnableLinearGrowthLimitation "True"',
            '.SetLinearGrowthLimitation ""',
            '.SingularEdgeRefinement "2"',
            '.DDMRefinementType "Automatic"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history('set 3d mesh adaptation properties', sCommand)

sCommand = ['Mesh.SetCreator "High Frequency"',
            'With FDSolver',
            '.Reset',
            '.SetMethod "Tetrahedral", "General purpose"',
            '.OrderTet "Second"',
            '.OrderSrf "First"',
            '.Stimulation "Zmax", "TE(0,0)"',
            '.ResetExcitationList',
            '.AddToExcitationList "Zmax", "TE(0,0);TM(0,0)"',
            '.AutoNormImpedance "False"',
            '.NormingImpedance "50"',
            '.ModesOnly "False"',
            '.ConsiderPortLossesTet "True"',
            '.SetShieldAllPorts "False"',
            '.AccuracyHex "1e-6"',
            '.AccuracyTet "1e-4"',
            '.AccuracySrf "1e-3"',
            '.LimitIterations "False"',
            '.MaxIterations "0"',
            '.SetCalcBlockExcitationsInParallel "True", "True", ""',
            '.StoreAllResults "False"',
            '.StoreResultsInCache "False"',
            '.UseHelmholtzEquation "True"',
            '.LowFrequencyStabilization "False"',
            '.Type "Auto"',
            '.MeshAdaptionHex "False"',
            '.MeshAdaptionTet "True"',
            '.AcceleratedRestart "True"',
            '.FreqDistAdaptMode "Distributed"',
            '.NewIterativeSolver "True"',
            '.TDCompatibleMaterials "False"',
            '.ExtrudeOpenBC "False"',
            '.SetOpenBCTypeHex "Default"',
            '.SetOpenBCTypeTet "Default"',
            '.AddMonitorSamples "True"',
            '.CalcPowerLoss "True"',
            '.CalcPowerLossPerComponent "False"',
            '.StoreSolutionCoefficients "True"',
            '.UseDoublePrecision "False"',
            '.UseDoublePrecision_ML "True"',
            '.MixedOrderSrf "False"',
            '.MixedOrderTet "False"',
            '.PreconditionerAccuracyIntEq "0.15"',
            '.MLFMMAccuracy "Default"',
            '.MinMLFMMBoxSize "0.3"',
            '.UseCFIEForCPECIntEq "True"',
            '.UseFastRCSSweepIntEq "false"',
            '.UseSensitivityAnalysis "False"',
            '.RemoveAllStopCriteria "Hex"',
            '.AddStopCriterion "All S-Parameters", "0.01", "2", "Hex", "True"',
            '.AddStopCriterion "Reflection S-Parameters", "0.01", "2", "Hex", "False"',
            '.AddStopCriterion "Transmission S-Parameters", "0.01", "2", "Hex", "False"',
            '.RemoveAllStopCriteria "Tet"',
            '.AddStopCriterion "All S-Parameters", "0.01", "2", "Tet", "False"',
            '.AddStopCriterion "Reflection S-Parameters", "0.01", "2", "Tet", "True"',
            '.AddStopCriterion "Transmission S-Parameters", "0.01", "2", "Tet", "False"',
            '.AddStopCriterion "All Probes", "0.05", "2", "Tet", "True"',
            '.RemoveAllStopCriteria "Srf"',
            '.AddStopCriterion "All S-Parameters", "0.01", "2", "Srf", "True"',
            '.AddStopCriterion "Reflection S-Parameters", "0.01", "2", "Srf", "False"',
            '.AddStopCriterion "Transmission S-Parameters", "0.01", "2", "Srf", "False"',
            '.SweepMinimumSamples "3"',
            '.SetNumberOfResultDataSamples "1001"',
            '.SetResultDataSamplingMode "Frequency (linear)"',
            '.SweepWeightEvanescent "1.0"',
            '.AccuracyROM "1e-4"',
            '.AddSampleInterval "", "", "1", "Automatic", "True"',
            '.AddSampleInterval "", "", "", "Automatic", "False"',
            '.MPIParallelization "False"',
            '.UseDistributedComputing "False"',
            '.NetworkComputingStrategy "RunRemote"',
            '.NetworkComputingJobCount "3"',
            '.UseParallelization "True"',
            '.MaxCPUs "128"',
            '.MaximumNumberOfCPUDevices "2"',
            'End With',
            'With IESolver',
            '.Reset',
            '.UseFastFrequencySweep "True"',
            '.UseIEGroundPlane "False"',
            '.SetRealGroundMaterialName ""',
            '.CalcFarFieldInRealGround "False"',
            '.RealGroundModelType "Auto"',
            '.PreconditionerType "Auto"',
            '.ExtendThinWireModelByWireNubs "False"',
            'End With',
            'With IESolver',
            '.SetFMMFFCalcStopLevel "0"',
            '.SetFMMFFCalcNumInterpPoints "6"',
            '.UseFMMFarfieldCalc "True"',
            '.SetCFIEAlpha "0.500000"',
            '.LowFrequencyStabilization "False"',
            '.LowFrequencyStabilizationML "True"',
            '.Multilayer "False"',
            '.SetiMoMACC_I "0.0001"',
            '.SetiMoMACC_M "0.0001"',
            '.DeembedExternalPorts "True"',
            '.SetOpenBC_XY "True"',
            '.OldRCSSweepDefintion "False"',
            '.SetRCSOptimizationProperties "True", "100", "0.00001"',
            '.SetAccuracySetting "Custom"',
            '.CalculateSParaforFieldsources "True"',
            '.ModeTrackingCMA "True"',
            '.NumberOfModesCMA "3"',
            '.StartFrequencyCMA "-1.0"',
            '.SetAccuracySettingCMA "Default"',
            '.FrequencySamplesCMA "0"',
            '.SetMemSettingCMA "Auto"',
            '.CalculateModalWeightingCoefficientsCMA "True"',
            'End With']
sCommand = line_break.join(sCommand)
modeler.add_to_history('sdefine frequency domain solver parameters', sCommand)

# 设置 PBA 版本
sCommand = 'Discretizer.PBAVersion "2019092520"'
modeler.add_to_history('set PBA version', sCommand)
# PBA版本设置完成

True

In [ ]:

count = 0       # 文件存储序号

#建模
for row in arr:

    count += 1
    a, b, l, w1, w2, s1, s2 = row[0], row[1], row[2], row[3], row[4], row[5], row[6]

    # 金属背板
    sCommand = ['With Brick',
                '.Reset',
                ' .Name "solid1"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange "-p/2", "p/2"',
                '.Yrange "-p/2", "p/2"',
                '.Zrange "0", "h1"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    # 介质层
    sCommand = ['With Brick',
                '.Reset',
                '.Name "solid2"',
                '.Component "component1"',
                '.Material "FR-4 (lossy)"',
                '.Xrange "-p/2", "p/2"',
                '.Yrange "-p/2", "p/2"',
                '.Zrange "h1", "h1+h2"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)


    # 创建金属单元
    # 横x，竖y
    # 十字型谐振器
    # 竖的窄长方形
    sCommand = ['With Brick',
                '.Reset',
                '.Name "1"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange "%f", "%f" '% (-l/2,l/2),
                '.Yrange "%f", "%f" '% (-w1/2,w1/2),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    sCommand = ['With Brick',
                '.Reset',
                '.Name "2"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange  "%f", "%f"'% (-w1/2,w1/2),
                '.Yrange  "%f", "%f"'% (-l/2,l/2),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    sCommand = ['Solid.Add "component1:1", "component1:2"']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('boolean add shapes', sCommand)          # 合并

    # 上半边，先定义两个金属正方块，之后会差掉中间的部分，成为环状
    sCommand = ['With Brick',
                '.Reset',
                '.Name "3"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange "%f", "%f" '%(-a-s1-w1/2, -s1-w1/2),
                '.Yrange  "%f", "%f"'%(s1+w1/2, s1+w1/2+a),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    sCommand = ['With Brick',
                '.Reset',
                '.Name "4"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange  "%f", "%f"'%(s1+w1/2, s1+w1/2+a),
                '.Yrange  "%f", "%f"'%(s1+w1/2, s1+w1/2+a),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

# 下半边两个正方块
    sCommand = ['With Brick',
                '.Reset',
                '.Name "5"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange "%f", "%f"'%(-b-s2-w1/2, -s2-w1/2),
                '.Yrange "%f", "%f"'%(-w1/2-s2-b, -w1/2-s2),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    sCommand = ['With Brick',
                '.Reset',
                '.Name "6"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange  "%f", "%f" '%(w1/2+s2, w1/2+s2+b),
                '.Yrange  "%f", "%f" '%(-w1/2-s2-b, -w1/2-s2),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    # 去掉每个正方块中间的部分，成为环状
    sCommand = ['With Brick',
                '.Reset',
                '.Name "3.1"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange  "%f", "%f"'%(-a-s1-w1/2+w2, -w2-s1-w1/2),
                '.Yrange  "%f", "%f" '%(w1/2+s1+w2, w1/2+s1+a-w2),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    sCommand = ['With Brick',
                '.Reset',
                '.Name "4.1"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange  "%f", "%f" '%(w1/2+s1+w2, w1/2+s1+a-w2),
                '.Yrange "%f", "%f"'%(w1/2+s1+w2, w1/2+s1+a-w2),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    sCommand = ['With Brick',
                '.Reset',
                '.Name "5.1"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange  "%f", "%f" '%(-b-s2-w1/2+w2, -w2-s2-w1/2),
                '.Yrange  "%f", "%f" '%(-w1/2-s2-b+w2, -w1/2-s2-w2),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    sCommand = ['With Brick',
                '.Reset',
                '.Name "6.1"',
                '.Component "component1"',
                '.Material "Copper (annealed)"',
                '.Xrange "%f", "%f" '%(w1/2+s2+w2, w1/2+s2+b-w2),
                '.Yrange "%f", "%f" '%(-w1/2-s2-b+w2, -w1/2-s2-w2),
                '.Zrange "h1+h2", "h1+h2+h3"',
                '.Create',
                'End With']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('define brick', sCommand)

    # 去掉中间部分，成为方环
    sCommand = ['Solid.Subtract "component1:3", "component1:3.1"']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('boolean subtract shapes', sCommand)

    sCommand = ['Solid.Subtract "component1:4", "component1:4.1"']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('boolean subtract shapes', sCommand)

    sCommand = ['Solid.Subtract "component1:5", "component1:5.1"']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('boolean subtract shapes', sCommand)

    sCommand = ['Solid.Subtract "component1:6", "component1:6.1"']
    sCommand = line_break.join(sCommand)
    modeler.add_to_history('boolean subtract shapes', sCommand)

    # 仿真开始
    modeler.run_solver()
    # 仿真结束

    # 导出dB数据
    sCommmd = [r'SelectTreeItem("1D Results\S-Parameters\SZmax(1),Zmax(1)")',
               'With Plot1D',
               '.PlotView "magnitudedb"',
               'End With',
               'With ASCIIExport',
               '.Reset',
               '.FileName "%s"' % rf'F:\metasurface_zwj\{count}-dB.txt',
               '.Execute',
               'End With']
    sCommmd = '\n'.join(sCommmd)
    modeler.add_to_history('save magnitudedb', sCommmd)


        # 删除component
    sCommand = 'Component.Delete "component1" '
    modeler.add_to_history('delete component', sCommand)

    # break
        # 删除完成